# UniDepth + hiện vật tự động + ByteTrack-DMA-LTC + touch candidate

Notebook này chạy trên ba video gốc: **Grounding DINO + SAM 2** tìm/mask hiện vật; **UniDepth** tạo depth 3D; **YOLOv8n person detector → ByteTrack-DMA-LTC → YOLOv8n-Pose trên crop theo track ID** tìm người và cổ tay. YOLO nano là trọng số detector/pose; DMA GBM và LTC checkpoint vẫn là trọng số tracker riêng. Một người chỉ được đánh dấu `TOUCH_CANDIDATE` khi cổ tay gần mask hiện vật trên ảnh, gần bề mặt theo depth 3D, và tín hiệu xuất hiện lặp lại ít nhất 3/5 frame.

Không cần vẽ polygon. Mask cũ của MuseumVisitors không được áp thẳng lên clip mới vì khác tọa độ/góc quay. Chọn GPU runtime Colab, chạy lần lượt các cell và upload `1.mp4`, `2.mp4`, `3.mp4`.

Đây là ứng viên cần review, **không phải xác nhận chạm**: monocular depth, pose cổ tay và detector hiện vật đều có sai số; ngưỡng mét chỉ dùng thử nghiệm, chưa hiệu chuẩn với đo đạc thật.

In [ ]:
!git clone -q https://github.com/lpiccinelli-eth/UniDepth.git /content/UniDepth
!pip -q install -e /content/UniDepth ultralytics scipy transformers accelerate lap cython_bbox lightgbm gdown loguru scikit-learn joblib
!git clone -q https://github.com/khangkaka066/ByteTrack-DMA-LTC-Motion-Tracker.git /content/ByteTrack-DMA-LTC-Motion-Tracker
!pip -q install -e /content/ByteTrack-DMA-LTC-Motion-Tracker --no-deps
import sys
sys.path.insert(0, '/content/ByteTrack-DMA-LTC-Motion-Tracker')
!pip -q install git+https://github.com/facebookresearch/sam2.git

In [ ]:
from google.colab import files
from pathlib import Path
uploaded=files.upload()  # upload 3 videos + DMA GBM + LTC checkpoint
required=('1.mp4','2.mp4','3.mp4','dma_gbm_mot17.gbm','ltc_motion_mot17.pth','isolation_forest.joblib','anomaly_scores.jsonl')
for n in required:
    if n not in uploaded: raise FileNotFoundError(f'Upload thiếu {n}')
    Path('/content',n).write_bytes(uploaded[n])
print('Đã nhận video, DMA/LTC và model hành vi T06; YOLOv8n/YOLOv8n-Pose tải tự động; không cần upload best_ckpt.zip.')

In [ ]:
import cv2, numpy as np, torch, csv, zipfile, json, inspect, subprocess, math, warnings, joblib
from pathlib import Path
from types import SimpleNamespace
from collections import defaultdict, deque
from PIL import Image, ImageDraw
from scipy.spatial import cKDTree
from scipy.optimize import linear_sum_assignment
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from sam2.sam2_image_predictor import SAM2ImagePredictor
from ultralytics import YOLO
from unidepth.models import UniDepthV2
from yolox.tracker.byte_tracker import BYTETracker
import unidepth.models.backbones.dinov2 as dinov2_backbone
dinov2_backbone.MemEffAttention=dinov2_backbone.Attention  # dùng PyTorch SDPA, tránh mismatch xFormers/CUDA
device='cuda' if torch.cuda.is_available() else 'cpu'
if device!='cuda': raise RuntimeError('Hãy chọn Runtime > Change runtime type > GPU')
W,H=640,400
WRIST_CONF_MIN=0.35; MAX_WRIST_ARTIFACT_GAP_M=0.30; MAX_IMAGE_GAP_PX=24
HISTORY=5; REQUIRED_HITS=3; RELEASE_MISSES=6
processor=AutoProcessor.from_pretrained('IDEA-Research/grounding-dino-base')
dino=AutoModelForZeroShotObjectDetection.from_pretrained('IDEA-Research/grounding-dino-base').to(device).eval()
sam=SAM2ImagePredictor.from_pretrained('facebook/sam2-hiera-small',device=device)
depth_model=UniDepthV2.from_pretrained('lpiccinelli/unidepth-v2-vits14').to(device).eval()
OUT=Path('/content/touch_distance_results'); OUT.mkdir(exist_ok=True)
REID=Path('/content/osnet_x1_0_imagenet.pth')
if not REID.exists(): subprocess.run(['gdown','-q','https://drive.google.com/uc?id=1LaG1EJpHrxdAxKnSCJ_i0u-nbxSAeiFY','-O',str(REID)],check=True)
if REID.stat().st_size<1000000: raise RuntimeError('Tải OSNet ReID weights không thành công.')
BEHAVIOR_BUNDLE=joblib.load('/content/isolation_forest.joblib')
BEHAVIOR_MODEL=BEHAVIOR_BUNDLE['pipeline']; SOURCE_FEATURE_NAMES=list(BEHAVIOR_BUNDLE['source_feature_names'])
ANOMALY_PERCENTILE_THRESHOLD=0.99  # same operating point as Museum/pose_rule_anomaly/configs/rule_thresholds.json
CALIBRATION={1:[],2:[],3:[]}
with open('/content/anomaly_scores.jsonl') as sf:
    for line in sf:
        row=json.loads(line)
        if row.get('evidence_status')=='scored' and row.get('split')=='train_pool' and row.get('anomaly_score') is not None:
            CALIBRATION[int(row['camera_id'])].append(float(row['anomaly_score']))
CALIBRATION={cam:np.sort(np.asarray(vals,np.float32)) for cam,vals in CALIBRATION.items()}
if any(len(CALIBRATION[cam])<30 for cam in (1,2,3)): raise RuntimeError('Thiếu score chuẩn hóa T06 cho một camera.')
KEYPOINT_NAMES=['nose','left_eye','right_eye','left_ear','right_ear','left_shoulder','right_shoulder','left_elbow','right_elbow','left_wrist','right_wrist','left_hip','right_hip','left_knee','right_knee','left_ankle','right_ankle']
def feature_names_t06():
    names=[f'pose_{name}_{axis}' for name in KEYPOINT_NAMES for axis in ('x','y')]
    names += [f'keypoint_confidence_{name}' for name in KEYPOINT_NAMES]
    names += [f'pose_velocity_{name}_{axis}' for name in KEYPOINT_NAMES for axis in ('x','y')]
    names += ['bbox_center_x','bbox_center_y','bbox_width','bbox_height','bbox_aspect','hip_height','torso_angle','bbox_velocity_x','bbox_velocity_y']
    return names
if feature_names_t06()!=SOURCE_FEATURE_NAMES: raise ValueError('T06 feature schema mismatches this notebook.')
def pose_features_t06(record):
    box=np.asarray(record['box'],np.float32); x1,y1,x2,y2=box; bw=max(float(x2-x1),1.0); bh=max(float(y2-y1),1.0)
    coords=np.asarray(record['keypoints'],np.float32); conf=np.asarray(record['confidence'],np.float32); visible=conf>=0.5
    hips=np.array([11,12]); shoulders=np.array([5,6]); origin=coords[hips].mean(axis=0) if visible[hips].all() else np.array([(x1+x2)/2,(y1+y2)/2],np.float32)
    scale=max(bh,math.sqrt(bw*bh),1.0); norm=(coords-origin)/scale; norm[~visible]=np.nan
    hip_height=np.nan; torso_angle=np.nan
    if visible[hips].all():
        hc=coords[hips].mean(axis=0); hip_height=(hc[1]-y1)/bh
        if visible[shoulders].all():
            v=coords[shoulders].mean(axis=0)-hc; torso_angle=math.atan2(float(v[0]),float(-v[1]))
    bbox=np.asarray([((x1+x2)/2)/W,((y1+y2)/2)/H,bw/W,bh/H,bw/bh,hip_height,torso_angle],np.float32)
    return norm.reshape(-1),conf,bbox
def build_t06_window(records,start_time,duration=2.5,n_steps=16):
    usable=[]
    for rec in records:
        vals=pose_features_t06(rec); usable.append((float(rec['time']),*vals))
    grid=np.linspace(start_time,start_time+duration,n_steps); pose=np.full((n_steps,34),np.nan,np.float32); confs=np.full((n_steps,17),np.nan,np.float32); bbox=np.full((n_steps,7),np.nan,np.float32); valid=np.zeros(n_steps,bool)
    if len(usable)<2: return None,{'valid_step_ratio':0.0,'keypoint_visibility':0.0}
    times=np.asarray([r[0] for r in usable])
    for i,target in enumerate(grid):
        right=int(np.searchsorted(times,target,side='left'))
        if right==0 or right==len(times): continue
        left=right-1; gap=times[right]-times[left]
        if gap<=0 or gap>0.75: continue
        frac=float((target-times[left])/gap)
        for dst,idx in ((pose,1),(confs,2),(bbox,3)):
            a,b=usable[left][idx],usable[right][idx]; ok=np.isfinite(a)&np.isfinite(b); dst[i,ok]=a[ok]+(b[ok]-a[ok])*frac
        valid[i]=np.isfinite(bbox[i,:5]).all()
    valid_ratio=float(valid.mean()); kp_mask=np.isfinite(pose.reshape(n_steps,17,2)).all(axis=2); visibility=float(kp_mask[valid].mean()) if valid.any() else 0.0
    quality={'valid_step_ratio':valid_ratio,'keypoint_visibility':visibility}
    if valid_ratio<0.90 or visibility<0.80: return None,quality
    dt=duration/(n_steps-1); pvel=np.full_like(pose,np.nan); pvel[1:]=(pose[1:]-pose[:-1])/dt; bvel=np.full((n_steps,2),np.nan,np.float32); bvel[1:]=(bbox[1:,:2]-bbox[:-1,:2])/dt
    return np.concatenate([pose,confs,pvel,bbox,bvel],axis=1),quality
def t06_embedding(window):
    keep=np.asarray([not n.startswith('keypoint_confidence_') for n in SOURCE_FEATURE_NAMES]); selected=window[:,keep]
    with warnings.catch_warnings():
        warnings.simplefilter('ignore',RuntimeWarning); med=np.nanmedian(selected,axis=0); spread=np.nanpercentile(selected,75,axis=0)-np.nanpercentile(selected,25,axis=0); low=np.nanmin(selected,axis=0); high=np.nanmax(selected,axis=0)
    return np.concatenate([med,spread,low,high]).astype(np.float32)[None,:]
def t06_percentile(camera_id,score):
    ref=CALIBRATION[int(camera_id)]; return float(np.searchsorted(ref,score,side='right')/max(len(ref),1))
def score_clip_tracks(clip,fps,frame_count,pose_history,window_rows,candidate_windows):
    camera_id=int(clip); duration=frame_count/max(fps,1e-6)
    for tid,records in pose_history.items():
        records.sort(key=lambda r:r['time']); start=math.ceil(records[0]['time'])
        while start+2.5<=duration+1e-6:
            window,quality=build_t06_window(records,start); end=start+2.5
            meta={'clip':clip,'camera_id':camera_id,'track_id':int(tid),'start_time':round(start,3),'end_time':round(end,3),'start_frame':int(round(start*fps)),'end_frame':int(round(end*fps)),'valid_step_ratio':quality['valid_step_ratio'],'keypoint_visibility':quality['keypoint_visibility']}
            if window is None: meta.update({'anomaly_score':None,'camera_percentile':None,'evidence_status':'insufficient_evidence','unusual_motion_candidate':False})
            else:
                score=float(-BEHAVIOR_MODEL.decision_function(t06_embedding(window))[0]); pct=t06_percentile(camera_id,score)
                meta.update({'anomaly_score':score,'camera_percentile':pct,'evidence_status':'scored','unusual_motion_candidate':pct>=ANOMALY_PERCENTILE_THRESHOLD})
                if meta['unusual_motion_candidate']: candidate_windows.append(dict(meta))
            window_rows.append(meta); start+=1.0
def merge_anomaly_candidates(rows,merge_gap=1.0):
    out=[]
    for clip,tid in sorted({(r['clip'],r['track_id']) for r in rows}):
        items=sorted([r for r in rows if r['clip']==clip and r['track_id']==tid],key=lambda r:r['start_time']); group=None
        for row in items:
            if group is None or row['start_time']>group['end_time']+merge_gap:
                if group: out.append(group)
                group={'clip':clip,'camera_id':row['camera_id'],'track_id':tid,'start_time':row['start_time'],'end_time':row['end_time'],'start_frame':row['start_frame'],'end_frame':row['end_frame'],'window_count':1,'max_anomaly_score':row['anomaly_score'],'max_camera_percentile':row['camera_percentile'],'min_valid_step_ratio':row['valid_step_ratio'],'min_keypoint_visibility':row['keypoint_visibility']}
            else:
                group['end_time']=max(group['end_time'],row['end_time']); group['end_frame']=max(group['end_frame'],row['end_frame']); group['window_count']+=1
                group['max_anomaly_score']=max(group['max_anomaly_score'],row['anomaly_score']); group['max_camera_percentile']=max(group['max_camera_percentile'],row['camera_percentile'])
                group['min_valid_step_ratio']=min(group['min_valid_step_ratio'],row['valid_step_ratio']); group['min_keypoint_visibility']=min(group['min_keypoint_visibility'],row['keypoint_visibility'])
        if group: out.append(group)
    for i,row in enumerate(out,1): row['event_id']=f"{row['clip']}_unusual_{i:03d}"; row['event_type']='unusual_motion_candidate'
    return out
def render_behavior_previews(events,track_rows,fps_by_clip):
    boxes={(r['clip'],r['frame'],r['track_id']):r['box'] for r in track_rows}
    for clip in ('1','2','3'):
        source=OUT/f'{clip}_touch_preview.mp4'; cap=cv2.VideoCapture(str(source)); fps=fps_by_clip[clip]; w=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); h=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        writer=cv2.VideoWriter(str(OUT/f'{clip}_behavior_preview.mp4'),cv2.VideoWriter_fourcc(*'mp4v'),fps,(w,h)); frame=0; relevant=[e for e in events if e['clip']==clip]
        while True:
            ok,img=cap.read()
            if not ok: break
            active=[e for e in relevant if e['start_frame']<=frame<=e['end_frame']]
            for e in active:
                box=boxes.get((clip,frame,e['track_id']))
                if box is not None: x1,y1,x2,y2=map(int,box); cv2.rectangle(img,(x1,y1),(x2,y2),(0,140,255),3)
                msg=f"UNUSUAL_MOTION_CANDIDATE {e['event_id']} ID {e['track_id']} p={e['max_camera_percentile']:.3f}"; cv2.rectangle(img,(5,36),(w-5,66),(0,95,180),-1); cv2.putText(img,msg,(12,57),cv2.FONT_HERSHEY_SIMPLEX,.42,(255,255,255),1,cv2.LINE_AA)
            writer.write(img); frame+=1
        cap.release(); writer.release()
tracker_args=SimpleNamespace(track_thresh=0.25,track_buffer=45,match_thresh=0.8,mot20=False,
    ltc_motion_ckpt='/content/ltc_motion_mot17.pth',ltc_history_len=16,ltc_input_dim=12,ltc_min_history=8,
    ltc_hidden_size=128,ltc_num_layers=2,ltc_device=device,ltc_covariance_scale=1.0,ltc_max_abs_residual=256.0,
    with_reid=True,reid_device=device,reid_weight=0.35,reid_thresh=0.7,reid_alpha=0.9,
    reid_model='osnet_x1_0',reid_model_path=str(REID),reid_backend='deep',fast_reid=False,
    fast_reid_config='',fast_reid_weights='',fast_reid_batch_size=16,ml='gbm',
    ml_weights='/content/dma_gbm_mot17.gbm',dma_weights='',dma_device=device)
CAPTION='a museum statue. a sculpture. a framed painting. a relief sculpture. a museum artwork. an exhibit.'
def box_iou(a,b):
    x1,y1=max(a[0],b[0]),max(a[1],b[1]); x2,y2=min(a[2],b[2]),min(a[3],b[3])
    inter=max(0,x2-x1)*max(0,y2-y1); area=lambda q:max(0,q[2]-q[0])*max(0,q[3]-q[1])
    return inter/max(area(a)+area(b)-inter,1e-6)
def run_tracker(tracker,dets,h,w,frame_no,frame):
    pars=inspect.signature(tracker.update).parameters
    if 'frame' in pars: return tracker.update(dets,(h,w,frame_no),(h,w),frame=frame)
    return tracker.update(dets,(h,w,frame_no),(h,w))
def assign_tracks(pboxes,tracks):
    if not len(pboxes) or not len(tracks): return {}
    mat=np.array([[box_iou(p,t) for t in tracks] for p in pboxes],np.float32)
    rr,cc=linear_sum_assignment(1.0-mat); return {int(r):int(c) for r,c in zip(rr,cc) if mat[r,c]>=0.20}
def make_person_crop(frame,box,pad_fraction=0.15):
    # Mở rộng crop một chút để không cắt mất cổ tay đang vươn khỏi thân người.
    x1,y1,x2,y2=np.asarray(box,np.float32); bw=max(1.0,x2-x1); bh=max(1.0,y2-y1)
    px,py=pad_fraction*bw,pad_fraction*bh
    cx1=max(0,int(np.floor(x1-px))); cy1=max(0,int(np.floor(y1-py)))
    cx2=min(frame.shape[1],int(np.ceil(x2+px))); cy2=min(frame.shape[0],int(np.ceil(y2+py)))
    crop=frame[cy1:cy2,cx1:cx2]
    if crop.size==0 or crop.shape[0]<4 or crop.shape[1]<4: return None
    local_box=np.asarray([x1-cx1,y1-cy1,x2-cx1,y2-cy1],np.float32)
    return crop,(cx1,cy1),local_box
def pose_for_crop_result(result,origin,target_box,frame_shape):
    if result.keypoints is None or result.boxes is None or len(result.boxes)==0: return None
    pred_boxes=result.boxes.xyxy.cpu().numpy().astype(np.float32)
    overlap=np.asarray([box_iou(b,target_box) for b in pred_boxes]); j=int(np.argmax(overlap))
    if overlap[j]<0.05: return None  # không nhận pose của người lọt vào crop bên cạnh
    xy=result.keypoints.xy.cpu().numpy(); conf=result.keypoints.conf.cpu().numpy() if result.keypoints.conf is not None else np.ones(xy.shape[:2],np.float32)
    ox,oy=origin; kp=xy[j].astype(np.float32)+np.asarray([ox,oy],np.float32)
    kp[:,0]=np.clip(kp[:,0],0,frame_shape[1]-1); kp[:,1]=np.clip(kp[:,1],0,frame_shape[0]-1)
    return kp,conf[j].astype(np.float32)
def find_artifacts(path):
    cap=cv2.VideoCapture(path); n=max(1,int(cap.get(cv2.CAP_PROP_FRAME_COUNT))); samples=[]
    for frac in (0.15,0.35,0.55,0.75,0.92):
        cap.set(cv2.CAP_PROP_POS_FRAMES,min(n-1,int((n-1)*frac))); ok,bgr=cap.read()
        if ok: samples.append(cv2.cvtColor(cv2.resize(bgr,(W,H)),cv2.COLOR_BGR2RGB))
    cap.release(); best=[]; ref=None
    for rgb in samples:
        inp=processor(images=Image.fromarray(rgb),text=CAPTION,return_tensors='pt').to(device)
        with torch.inference_mode(): out=dino(**inp)
        try: res=processor.post_process_grounded_object_detection(out,threshold=0.22,text_threshold=0.20,target_sizes=[(H,W)])[0]
        except TypeError: res=processor.post_process_grounded_object_detection(out,inp.input_ids,box_threshold=0.22,text_threshold=0.20,target_sizes=[(H,W)])[0]
        boxes=res.get('boxes',[]); scores=res.get('scores',[]); labels=res.get('text_labels',res.get('labels',['']*len(boxes))); cur=[]
        for j,b in enumerate(boxes):
            q=[float(x) for x in b.detach().cpu().tolist()]; score=float(scores[j].detach().cpu()); label=str(labels[j]) if j<len(labels) else 'artwork'
            area=max(0,q[2]-q[0])*max(0,q[3]-q[1])/(W*H)
            if score>=0.22 and 0.0003<=area<=0.55: cur.append({'box':q,'score':score,'label':label})
        if len(cur)>len(best): best=cur; ref=rgb
    kept=[]
    for c in sorted(best,key=lambda x:x['score'],reverse=True):
        if all(box_iou(c['box'],k['box'])<0.55 for k in kept): kept.append(c)
        if len(kept)>=12: break
    if not kept: raise RuntimeError(f'{path}: không tìm thấy hiện vật; dừng để tránh gắn nhãn touch sai.')
    sam.set_image(ref); items=[]
    for j,c in enumerate(kept):
        with torch.inference_mode(): masks,_,_=sam.predict(point_coords=None,point_labels=None,box=np.asarray(c['box'],np.float32)[None,:],multimask_output=False)
        mask=np.asarray(masks).squeeze().astype(bool)
        if mask.shape!=(H,W): mask=cv2.resize(mask.astype(np.uint8),(W,H),interpolation=cv2.INTER_NEAREST)>0
        items.append({'artifact_id':f'artifact_{j+1}','label':c['label'],'score':c['score'],'mask':mask,'box':c['box']})
    return ref,items
def get_depth_K(rgb):
    inp=torch.from_numpy(rgb.copy()).permute(2,0,1).to(device)
    with torch.inference_mode(): pred=depth_model.infer(inp)
    depth=pred['depth'].squeeze().detach().float().cpu().numpy(); K=pred['intrinsics'].squeeze().detach().float().cpu().numpy()[:3,:3]
    dh,dw=depth.shape; depth=cv2.resize(depth,(W,H),interpolation=cv2.INTER_AREA); K=np.diag([W/dw,H/dh,1.0]).astype(np.float32)@K
    return depth.astype(np.float32),K
def point3d(depth,K,x,y):
    x=int(np.clip(round(x),0,W-1)); y=int(np.clip(round(y),0,H-1)); patch=depth[max(0,y-2):min(H,y+3),max(0,x-2):min(W,x+3)]
    valid=patch[np.isfinite(patch)&(patch>0.05)&(patch<100)]
    if not len(valid): return None
    ray=np.linalg.inv(K).astype(np.float32)@np.array([x,y,1.0],np.float32)
    return ray*float(np.median(valid))


In [ ]:
touch_rows=[]; distance_rows=[]; event_rows=[]; anomaly_window_rows=[]; anomaly_window_candidates=[]; track_rows=[]; fps_by_clip={}; global_event_no=0
palette=[(40,220,80),(255,170,20),(230,70,210),(40,210,240),(180,120,255)]
# Nano RGB models: detector first, then pose only on crops belonging to tracked IDs.
person_detector=YOLO('yolov8n.pt')
pose_model=YOLO('yolov8n-pose.pt')
for cid in ('1','2','3'):
    path=f'/content/{cid}.mp4'; ref,arts=find_artifacts(path)
    # Lưu mask nhị phân + ảnh kiểm tra Grounding DINO/SAM2.
    Image.fromarray(ref).save(OUT/f'{cid}_artifact_reference.jpg')
    np.savez_compressed(OUT/f'{cid}_artifact_masks.npz',**{a['artifact_id']:a['mask'].astype(np.uint8) for a in arts})
    review=Image.fromarray(ref).convert('RGBA'); tint=np.zeros((H,W,4),np.uint8); dr=ImageDraw.Draw(review)
    for j,a in enumerate(arts):
        tint[a['mask']]=(*palette[j%len(palette)],105); dr.rectangle(a['box'],outline='yellow',width=2); dr.text((a['box'][0],max(0,a['box'][1]-14)),f"{a['artifact_id']} {a['label']} {a['score']:.2f}",fill='yellow')
    Image.alpha_composite(review,Image.fromarray(tint)).convert('RGB').save(OUT/f'{cid}_artifact_mask_review.jpg')
    dtrans=[cv2.distanceTransform((~a['mask']).astype(np.uint8),cv2.DIST_L2,3) for a in arts]
    artifact_trees=[]
    for a in arts:
        yy,xx=np.where(a['mask'] & np.isfinite(np.ones((H,W),np.float32)))
        pix=np.stack([xx,yy,np.ones_like(xx)],axis=0).astype(np.float32); rays=np.linalg.inv(np.eye(3,dtype=np.float32))@pix
        # K varies by frame; point cloud is rebuilt per frame below from the mask pixels.
        artifact_trees.append((xx,yy))
    cap=cv2.VideoCapture(path); fps=cap.get(cv2.CAP_PROP_FPS); fps=fps if np.isfinite(fps) and fps>1 else 24.0
    person_tracker=BYTETracker(tracker_args,frame_rate=fps)  # DMA+LTC track state resets per clip
    writer=cv2.VideoWriter(str(OUT/f'{cid}_touch_preview.mp4'),cv2.VideoWriter_fourcc(*'mp4v'),fps,(W,H))
    fps_by_clip[cid]=fps; pose_history=defaultdict(list); states={}; frame_no=0
    while True:
        ok,bgr=cap.read()
        if not ok: break
        small=cv2.resize(bgr,(W,H)); rgb=cv2.cvtColor(small,cv2.COLOR_BGR2RGB); depth,K=get_depth_K(rgb); invK=np.linalg.inv(K).astype(np.float32)
        # Artifact point clouds in this frame's camera coordinate system.
        trees=[]
        for ai,a in enumerate(arts):
            xx,yy=artifact_trees[ai]; z=depth[yy,xx]; good=np.isfinite(z)&(z>0.05)&(z<100); xx2,yy2,z=xx[good],yy[good],z[good]
            if len(xx2)>12000: take=np.linspace(0,len(xx2)-1,12000).astype(int); xx2,yy2,z=xx2[take],yy2[take],z[take]
            pix=np.stack([xx2,yy2,np.ones_like(xx2)],axis=0).astype(np.float32); pts=(invK@pix*z[None,:]).T if len(xx2) else np.empty((0,3),np.float32)
            trees.append(cKDTree(pts) if len(pts) else None)
        det_result=person_detector.predict(small,classes=[0],conf=0.25,imgsz=640,verbose=False)[0]
        current={}; canvas=small.copy(); detections=[]
        if det_result.boxes is not None and len(det_result.boxes):
            det_boxes=det_result.boxes.xyxy.cpu().numpy().astype(np.float32)
            det_confs=det_result.boxes.conf.cpu().numpy().astype(np.float32)
        else:
            det_boxes=np.empty((0,4),np.float32); det_confs=np.empty((0,),np.float32)
        dets=np.column_stack([det_boxes,det_confs]).astype(np.float32) if len(det_boxes) else np.empty((0,5),np.float32)
        # Bước 1: tracker nhận person detections và cấp ID trước khi chạy pose.
        tracked=run_tracker(person_tracker,dets,H,W,frame_no,small)
        track_boxes=[np.asarray(t.tlbr,np.float32) for t in tracked]; track_ids=[int(t.track_id) for t in tracked]
        match=assign_tracks(det_boxes,track_boxes)  # detection index -> track index
        pose_jobs=[]
        for pi,ti in match.items():
            crop_info=make_person_crop(small,det_boxes[pi])
            if crop_info is None: continue
            crop,origin,local_box=crop_info
            pose_jobs.append({'crop':crop,'origin':origin,'local_box':local_box,'box':det_boxes[pi],'track_id':track_ids[ti]})
        # Bước 2: pose chạy theo batch trên các crop đã có track ID.
        pose_results=pose_model.predict([j['crop'] for j in pose_jobs],classes=[0],conf=0.10,imgsz=640,verbose=False) if pose_jobs else []
        for job,pose_result in zip(pose_jobs,pose_results):
            pose_data=pose_for_crop_result(pose_result,job['origin'],job['local_box'],small.shape)
            if pose_data is None: continue
            kp,kc=pose_data; box=job['box']; tid=job['track_id']
            pose_history[int(tid)].append({'time':frame_no/fps,'frame':frame_no,'box':box.astype(np.float32).copy(),'keypoints':kp.astype(np.float32).copy(),'confidence':kc.astype(np.float32).copy()})
            track_rows.append({'clip':cid,'frame':frame_no,'track_id':int(tid),'box':box.astype(np.float32).tolist()})
            bh=max(1,float(box[3]-box[1])); best=None
            for wrist_idx in (9,10):
                x,y=kp[wrist_idx]; conf=float(kc[wrist_idx])
                if conf<WRIST_CONF_MIN: continue
                w3=point3d(depth,K,x,y)
                if w3 is None: continue
                for ai,a in enumerate(arts):
                    tree=trees[ai]
                    if tree is None: continue
                    gap3=float(tree.query(w3,k=1)[0]); xi=int(np.clip(round(x),0,W-1)); yi=int(np.clip(round(y),0,H-1)); gap2=float(dtrans[ai][yi,xi])
                    pixel_limit=max(12.0,min(MAX_IMAGE_GAP_PX,0.08*bh)); qualifies=(gap3<=MAX_WRIST_ARTIFACT_GAP_M and gap2<=pixel_limit)
                    distance_rows.append({'clip':cid,'frame':frame_no,'time_s':round(frame_no/fps,3),'track_id':int(tid),'artifact_id':a['artifact_id'],'wrist':'left' if wrist_idx==9 else 'right','wrist_confidence':round(conf,3),'image_gap_px':round(gap2,2),'estimated_3d_gap_m':round(gap3,3),'touch_frame_gate':bool(qualifies)})
                    key=(int(tid),a['artifact_id']); current[key]=current.get(key,False) or qualifies
                    if best is None or gap3<best[0]: best=(gap3,gap2,a,wrist_idx,x,y,conf,qualifies)
            x1,y1,x2,y2=box.astype(int); cv2.rectangle(canvas,(x1,y1),(x2,y2),(80,180,255),1)
            cv2.putText(canvas,f'ID {int(tid)}',(x1,max(14,y1-3)),cv2.FONT_HERSHEY_SIMPLEX,.42,(255,255,0),1,cv2.LINE_AA)
            if best:
                gap3,gap2,a,wi,x,y,conf,qualifies=best; cv2.circle(canvas,(int(x),int(y)),4,(0,255,255),-1)
        # Temporal gate: at least 3 qualifying observations in the last 5 frames.
        for key,st in list(states.items()):
            hit=current.get(key,False); st['history'].append(hit)
            if sum(st['history'])>=REQUIRED_HITS:
                if not st['active']:
                    global_event_no+=1; st['active']=True; st['event_id']=f'{cid}_touch_{global_event_no:03d}'; st['start']=frame_no; st['min_gap']=999.0
                st['last_hit']=frame_no
            if st['active'] and frame_no-st['last_hit']>RELEASE_MISSES:
                event_rows.append({'event_id':st['event_id'],'clip':cid,'track_id':key[0],'artifact_id':key[1],'start_frame':st['start'],'end_frame':st['last_hit'],'duration_s':round((st['last_hit']-st['start'])/fps,3),'status':'touch_candidate_needs_visual_review','minimum_estimated_wrist_gap_m':round(st['min_gap'],3)})
                st['active']=False
        for key,hit in current.items():
            if key not in states: states[key]={'history':deque(maxlen=HISTORY),'active':False,'last_hit':-999,'min_gap':999.0}
            st=states[key]
            if hit:
                vals=[r['estimated_3d_gap_m'] for r in distance_rows[-20:] if r['clip']==cid and r['track_id']==key[0] and r['artifact_id']==key[1] and r['touch_frame_gate']]
                if vals: st['min_gap']=min(st['min_gap'],min(vals))
            if st['active']:
                label=f"TOUCH_CANDIDATE {st['event_id']} ID {key[0]} -> {key[1]}"
                cv2.rectangle(canvas,(6,6),(W-6,32),(0,0,180),-1); cv2.putText(canvas,label,(12,25),cv2.FONT_HERSHEY_SIMPLEX,.48,(255,255,255),1,cv2.LINE_AA)
        tint=np.zeros_like(canvas)
        for ai,a in enumerate(arts): tint[a['mask']]=palette[ai%len(palette)]
        canvas=cv2.addWeighted(canvas,0.82,tint,0.18,0); writer.write(canvas); frame_no+=1
    cap.release(); writer.release()
    for key,st in states.items():
        if st['active']: event_rows.append({'event_id':st['event_id'],'clip':cid,'track_id':key[0],'artifact_id':key[1],'start_frame':st['start'],'end_frame':st['last_hit'],'duration_s':round((st['last_hit']-st['start'])/fps,3),'status':'touch_candidate_needs_visual_review','minimum_estimated_wrist_gap_m':round(st['min_gap'],3)})
    score_clip_tracks(cid,fps,frame_no,pose_history,anomaly_window_rows,anomaly_window_candidates)
    print(f'Clip {cid}: {len(arts)} mask hiện vật, {frame_no} frames đã xử lý')
anomaly_events=merge_anomaly_candidates(anomaly_window_candidates)
render_behavior_previews(anomaly_events,track_rows,fps_by_clip)
for ev in event_rows:
    ctx=[r['camera_percentile'] for r in anomaly_window_rows if r['clip']==ev['clip'] and r['track_id']==ev['track_id'] and r['evidence_status']=='scored' and r['start_time']<=ev['end_frame']/fps_by_clip[ev['clip']] and r['end_time']>=ev['start_frame']/fps_by_clip[ev['clip']]]
    ev['peak_anomaly_percentile']=max(ctx) if ctx else None
def save_csv(path,rows,fields):
    with path.open('w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=fields); w.writeheader(); w.writerows(rows)
save_csv(OUT/'wrist_distance_by_frame.csv',distance_rows,['clip','frame','time_s','track_id','artifact_id','wrist','wrist_confidence','image_gap_px','estimated_3d_gap_m','touch_frame_gate'])
save_csv(OUT/'touch_candidates.csv',event_rows,['event_id','clip','track_id','artifact_id','start_frame','end_frame','duration_s','status','minimum_estimated_wrist_gap_m'])
save_csv(OUT/'anomaly_windows.csv',anomaly_window_rows,['clip','camera_id','track_id','start_time','end_time','start_frame','end_frame','valid_step_ratio','keypoint_visibility','anomaly_score','camera_percentile','evidence_status','unusual_motion_candidate'])
save_csv(OUT/'anomaly_candidates.csv',anomaly_events,['event_id','event_type','clip','camera_id','track_id','start_time','end_time','start_frame','end_frame','window_count','max_anomaly_score','max_camera_percentile','min_valid_step_ratio','min_keypoint_visibility'])
save_csv(OUT/'touch_candidates.csv',event_rows,['event_id','clip','track_id','artifact_id','start_frame','end_frame','duration_s','status','minimum_estimated_wrist_gap_m','peak_anomaly_percentile'])
print('Touch candidate events:',len(event_rows),'| unusual motion candidates:',len(anomaly_events))

In [ ]:
archive='/content/touch_distance_unidepth_auto_artifacts_results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.iterdir(): z.write(p,p.name)
files.download(archive)